## librerias

In [6]:
import pandas as pd
import numpy as np
import pyreadr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.svm import SVC


## carga de datos

In [7]:
file_path = "../data/listings.RData"

result = pyreadr.read_r(file_path)

df = result[list(result.keys())[0]]

df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


## exploracion y extraccion

In [8]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171748 entries, 0 to 171747
Data columns (total 80 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   id                                            171748 non-null  float64
 1   listing_url                                   171748 non-null  object 
 2   scrape_id                                     171748 non-null  float64
 3   last_scraped                                  171748 non-null  object 
 4   source                                        171748 non-null  object 
 5   name                                          171748 non-null  object 
 6   description                                   171748 non-null  object 
 7   neighborhood_overview                         171748 non-null  object 
 8   picture_url                                   171748 non-null  object 
 9   host_id                                       17

id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                              0
                                                ...  
calculated_host_listings_count_entire_homes         0
calculated_host_listings_count_private_rooms        0
calculated_host_listings_count_shared_rooms         0
reviews_per_month                               40287
city                                                0
Length: 80, dtype: int64

In [9]:
## limpiar datos 

df['price'] = df['price'].astype(str)
df['price'] = df['price'].str.replace(r"[^\d.]", "", regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df = df.dropna(subset=['price'])
df = df.reset_index(drop=True)

## variables categoricas

In [10]:
df['price_category'] = pd.qcut(
    df['price'],
    q=3,
    labels=['barata', 'media', 'cara']
)

In [11]:
# SOLO variables numéricas
X = df.select_dtypes(include=['int64', 'float64'])

X = X.drop(columns=['price'], errors='ignore')

y = df['price_category']

In [12]:
# Eliminar columnas con demasiados nulos (opcional pero recomendado)
X = X.dropna(axis=1, thresh=len(X)*0.7)

# Rellenar valores faltantes con la media
X = X.fillna(X.mean())

## division de datos (train/test)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## modelos svm

In [15]:
svm_linear = SVC(kernel='linear', C=1)

svm_linear.fit(X_train, y_train)

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [16]:
svm_rbf = SVC(kernel='rbf', C=1, gamma='scale')

svm_rbf.fit(X_train, y_train)

,C,1
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [17]:
svm_poly = SVC(kernel='poly', degree=3, C=1)

svm_poly.fit(X_train, y_train)

,C,1
,kernel,'poly'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


## predicciones 

In [18]:
y_pred_linear = svm_linear.predict(X_test)
y_pred_rbf = svm_rbf.predict(X_test)
y_pred_poly = svm_poly.predict(X_test)

## Matriz de confusion 

In [19]:
print("=== SVM LINEAR ===")
print(confusion_matrix(y_test, y_pred_linear))
print(classification_report(y_test, y_pred_linear))

print("=== SVM RBF ===")
print(confusion_matrix(y_test, y_pred_rbf))
print(classification_report(y_test, y_pred_rbf))

print("=== SVM POLY ===")
print(confusion_matrix(y_test, y_pred_poly))
print(classification_report(y_test, y_pred_poly))

=== SVM LINEAR ===
[[3315  382 1441]
 [ 665 3516  900]
 [1652 1477 1902]]
              precision    recall  f1-score   support

      barata       0.59      0.65      0.62      5138
        cara       0.65      0.69      0.67      5081
       media       0.45      0.38      0.41      5031

    accuracy                           0.57     15250
   macro avg       0.56      0.57      0.57     15250
weighted avg       0.56      0.57      0.57     15250

=== SVM RBF ===
[[3687  264 1187]
 [ 404 3336 1341]
 [1477 1084 2470]]
              precision    recall  f1-score   support

      barata       0.66      0.72      0.69      5138
        cara       0.71      0.66      0.68      5081
       media       0.49      0.49      0.49      5031

    accuracy                           0.62     15250
   macro avg       0.62      0.62      0.62     15250
weighted avg       0.62      0.62      0.62     15250

=== SVM POLY ===
[[3403  323 1412]
 [ 449 3380 1252]
 [1456 1246 2329]]
              precisi

## 7. Análisis de Sobreajuste y Desajuste

In [20]:
print("=== OVERFITTING ANALYSIS ===")

print("Linear Train:", svm_linear.score(X_train, y_train))
print("Linear Test:", svm_linear.score(X_test, y_test))

print("RBF Train:", svm_rbf.score(X_train, y_train))
print("RBF Test:", svm_rbf.score(X_test, y_test))

print("Poly Train:", svm_poly.score(X_train, y_train))
print("Poly Test:", svm_poly.score(X_test, y_test))

=== OVERFITTING ANALYSIS ===
Linear Train: 0.5714637025378714
Linear Test: 0.5726557377049181
RBF Train: 0.6380746278444488
RBF Test: 0.6224918032786885
Poly Train: 0.6087940192799528
Poly Test: 0.5975081967213115


## 8.

In [21]:
import time

# Linear
start = time.time()
svm_linear.fit(X_train, y_train)
time_linear = time.time() - start

# RBF
start = time.time()
svm_rbf.fit(X_train, y_train)
time_rbf = time.time() - start

# Poly
start = time.time()
svm_poly.fit(X_train, y_train)
time_poly = time.time() - start

print("Tiempo Linear:", time_linear)
print("Tiempo RBF:", time_rbf)
print("Tiempo Poly:", time_poly)

Tiempo Linear: 436.165646314621
Tiempo RBF: 288.0778069496155
Tiempo Poly: 550.0759615898132


In [22]:

resultados = pd.DataFrame({
    "Modelo": ["Linear", "RBF", "Poly"],
    "Train Accuracy": [
        svm_linear.score(X_train, y_train),
        svm_rbf.score(X_train, y_train),
        svm_poly.score(X_train, y_train)
    ],
    "Test Accuracy": [
        svm_linear.score(X_test, y_test),
        svm_rbf.score(X_test, y_test),
        svm_poly.score(X_test, y_test)
    ],
    "Tiempo": [time_linear, time_rbf, time_poly]
})

print(resultados)

   Modelo  Train Accuracy  Test Accuracy      Tiempo
0  Linear        0.571464       0.572656  436.165646
1     RBF        0.638075       0.622492  288.077807
2    Poly        0.608794       0.597508  550.075962


## 10.

In [25]:
X = df.select_dtypes(include=['int64', 'float64']).drop("price", axis=1)
y = df["price"]

In [29]:
from sklearn.model_selection import train_test_split
X = X.fillna(X.mean())
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [31]:
from sklearn.svm import SVR

svr = SVR(kernel='rbf', C=100, gamma=0.1)
svr.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,0.1
,coef0,0.0
,tol,0.001
,C,100
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = svr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 597.0271107711776
RMSE: 4323.9369172236875
